# Analyst Workbench

Este cuaderno te deja escoger un `ticker`, una `fecha`, un `agente` y una `peticion`.

El flujo hace esto:
- te muestra que data requiere el agente elegido
- corre automaticamente los prerequisitos necesarios
- ejecuta la etapa del agente con los modelos del repo
- genera una respuesta final orientada a tu peticion
- guarda artefactos en disco para que puedas inspeccionarlos

Importante:
- No expone chain-of-thought privado del modelo.
- Si falta data suficiente, la respuesta debe indicarlo explicitamente.

In [ ]:
from pathlib import Path
import json
from pprint import pprint
from IPython.display import Markdown, display

from analyst_access.workbench import describe_agent, list_agents, run_agent_request

TICKER = 'QQQ'
TRADE_DATE = '2026-03-23'
AGENT_KEY = 'market_analyst'
USER_REQUEST = 'Analiza el ticker y dime si ves una idea tactica interesante para swing trading. Si la data no alcanza, dime exactamente que falta.'

# Opcional: usar los defaults del repo dejando None.
QUICK_MODEL = None
DEEP_MODEL = None
REASONING_EFFORT = 'low'

OUTPUT_ROOT = 'analyst_access_sessions'


In [ ]:
display(Markdown('## Agentes disponibles'))
for item in list_agents():
    print(f"- {item['agent_key']}: {item['display_name']}")
    print('  prereqs:', item['prerequisites'])
    for needed in item['required_data']:
        print('   *', needed)
    print()

In [ ]:
spec = describe_agent(AGENT_KEY)
display(Markdown(f"## Agente seleccionado: `{AGENT_KEY}`"))
pprint(spec)

In [ ]:
result = run_agent_request(
    ticker=TICKER,
    trade_date=TRADE_DATE,
    agent_key=AGENT_KEY,
    user_request=USER_REQUEST,
    output_root=OUTPUT_ROOT,
    quick_model=QUICK_MODEL,
    deep_model=DEEP_MODEL,
    reasoning_effort=REASONING_EFFORT,
)

display(Markdown('## Resultado de la sesion'))
pprint({k: v for k, v in result.items() if k != 'answer'})
display(Markdown('## Respuesta del agente'))
display(Markdown(result['answer']))

In [ ]:
session_dir = Path(result['output_dir'])
display(Markdown(f'## Carpeta de salida\n`{session_dir}`'))

for path in sorted(session_dir.iterdir()):
    print(path.name)

In [ ]:
summary_path = session_dir / 'session_summary.json'
if summary_path.exists():
    display(Markdown('## Session summary'))
    pprint(json.loads(summary_path.read_text(encoding='utf-8')))

answer_path = session_dir / 'selected_answer.md'
if answer_path.exists():
    display(Markdown('## Respuesta guardada en disco'))
    display(Markdown(answer_path.read_text(encoding='utf-8')))

In [ ]:
display(Markdown('## Etapas ejecutadas y archivos generados'))
for path in sorted([p for p in session_dir.iterdir() if p.is_dir()]):
    print(f'[{path.name}]')
    for child in sorted(path.iterdir()):
        print('  -', child.name)
    print()